In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='groq:qwen/qwen3-32b',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='groq:qwen/qwen3-32b',
    tools=[square]
)

In [4]:
from langchain.messages import HumanMessage
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

main_agent = create_agent(model='groq:qwen/qwen3-32b'
,tools = [call_subagent_1,call_subagent_2],
system_prompt = "you are a helpful assistant who can call subagents to calculate the square root or square of a number."
)


In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='60286108-768d-4f31-9cfb-5f5338ea2ba4'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the square root of 456. Let me see. I need to figure out which subagent to use. The tools provided are call_subagent_1 for square roots and call_subagent_2 for squares. Since they want the square root, I should use the first subagent. The parameter required is x, which is 456 in this case. I'll make sure to structure the tool call correctly with the function name and arguments in JSON. Let me double-check the syntax to avoid any errors.\n", 'tool_calls': [{'id': 's1awpawym', 'function': {'arguments': '{"x":456}', 'name': 'call_subagent_1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 257, 'total_tokens': 399, 'completion_time': 0.24053756, 'completion_tokens_details': {'